In [1]:
!pip install timm -q

In [11]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.signal import stft
import warnings
warnings.filterwarnings('ignore')

DATA_DIR    = Path('.')
TRAIN_H5    = DATA_DIR / '/kaggle/input/competitions/ai-team-challenge-3/train_dataset.h5'
TEST_H5     = DATA_DIR / '/kaggle/input/competitions/ai-team-challenge-3/test_dataset.h5'
SUBMIT_CSV  = DATA_DIR / '/kaggle/input/competitions/ai-team-challenge-3/sample_submission.csv'

IMG_SIZE    = 224
SAMPLE_RATE = 4096
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 3e-4
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

Device: cuda


In [17]:
def inspect_h5(path: Path):
    print(f'=== Ouverture de {path} ===')
    with h5py.File(path, 'r') as f:
        print('Clés à la racine :', list(f.keys()))
        for key in f.keys():
            print(f" - {key} : shape={f[key].shape}, dtype={f[key].dtype}")

# Exécution de l'exploration sur le H5 :
inspect_h5(TRAIN_H5)

=== Ouverture de /kaggle/input/competitions/ai-team-challenge-3/train_dataset.h5 ===
Clés à la racine : ['X', 'id', 'snr', 'y']
 - X : shape=(14000, 4096), dtype=float32
 - id : shape=(14000,), dtype=|S14
 - snr : shape=(14000,), dtype=float32
 - y : shape=(14000,), dtype=int32


In [18]:
def signal_to_spectrogram(signal: np.ndarray, fs: int = SAMPLE_RATE, img_size: int = IMG_SIZE) -> np.ndarray:
    from PIL import Image
    signal = signal.astype(np.float32)
    
    # 1. Nettoyage des valeurs NaN et Inf (remplacées par 0)
    signal = np.nan_to_num(signal, nan=0.0, posinf=0.0, neginf=0.0)
    
    # 2. Sécurité : éviter la division par zéro si le signal est vide/constant
    signal_std = signal.std()
    if signal_std == 0:
        signal_std = 1.0
        
    signal = (signal - signal.mean()) / (signal_std + 1e-8)

    _, _, Zxx = stft(signal, fs=fs, nperseg=256, noverlap=224, boundary=None)
    magnitude_db = 20 * np.log10(np.abs(Zxx) + 1e-9)
    
    # 3. Re-nettoyage post-spectrogramme au cas où
    magnitude_db = np.nan_to_num(magnitude_db, nan=0.0, posinf=0.0, neginf=0.0)
    
    mn, mx = magnitude_db.min(), magnitude_db.max()
    if mx == mn:
        img = np.zeros_like(magnitude_db)
    else:
        img = (magnitude_db - mn) / (mx - mn + 1e-8)

    pil_img = Image.fromarray((img * 255).astype(np.uint8))
    pil_img = pil_img.resize((img_size, img_size), Image.BILINEAR)
    return np.array(pil_img).astype(np.float32) / 255.0

print('Fonction de transformation avec nettoyage NaN/Inf OK.')

Fonction de transformation avec nettoyage NaN/Inf OK.


In [19]:
class GWDataset(Dataset):
    TRAIN_TRANSFORM = T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.ColorJitter(brightness=0.15, contrast=0.15),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    VAL_TRANSFORM = T.Compose([
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, h5_path, indices, mode='train'):
        """
        mode: 'train', 'val', ou 'test'
        """
        self.h5_path   = str(h5_path)
        self.indices   = indices
        self.mode      = mode
        self.transform = self.TRAIN_TRANSFORM if mode == 'train' else self.VAL_TRANSFORM

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        with h5py.File(self.h5_path, 'r') as f:
            signal = f['X'][real_idx]
            
            if self.mode == 'test':
                sample_id = f['id'][real_idx]
                if hasattr(sample_id, 'decode'):
                    sample_id = sample_id.decode('utf-8')
            else:
                label = f['y'][real_idx]

        # Si le signal a plusieurs canaux (ex: 2 détecteurs), on moyenne
        if signal.ndim > 1:
            signal = signal.mean(axis=0)

        spec = signal_to_spectrogram(signal)
        spec_rgb = np.stack([spec, spec, spec], axis=0)
        tensor = torch.tensor(spec_rgb, dtype=torch.float32)
        tensor = self.transform(tensor)

        if self.mode == 'test':
            return tensor, sample_id
        return tensor, torch.tensor(label, dtype=torch.float32)

print('Classe GWDataset OK.')

Classe GWDataset OK.


In [20]:
class GWClassifier(nn.Module):
    def __init__(self, backbone='efficientnet_b3'):
        super().__init__()
        base = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(p=0.35), 
            nn.Linear(in_features, 1)
        )
        self.model = base

    def forward(self, x):
        return self.model(x).squeeze(1)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    for imgs, labels in loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
        preds.extend(probs)
        targets.extend(labels.numpy())
    return roc_auc_score(targets, preds), np.array(preds)

In [21]:
print('Chargement des indices et des labels pour le split...')
with h5py.File(TRAIN_H5, 'r') as f:
    num_samples = f['X'].shape[0]
    all_labels = f['y'][:]

indices = np.arange(num_samples)
tr_idx, val_idx, tr_labels, val_labels = train_test_split(
    indices, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)
print(f'Train: {len(tr_idx)}  |  Val: {len(val_idx)}')

train_loader = DataLoader(GWDataset(TRAIN_H5, tr_idx, mode='train'), BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(GWDataset(TRAIN_H5, val_idx, mode='val'), BATCH_SIZE, shuffle=False)

model     = GWClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()

best_auc = 0.0
print('\n--- Début des Epochs ---')
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    auc, _  = evaluate(model, val_loader)
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS} -> loss: {loss:.4f} | AUC: {auc:.4f}')
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'   [+] Nouveau meilleur modèle (AUC={best_auc:.4f}) mis de côté !')

Chargement des indices et des labels pour le split...
Train: 11200  |  Val: 2800

--- Début des Epochs ---
Epoch 01/10 -> loss: 0.1305 | AUC: 0.9712
   [+] Nouveau meilleur modèle (AUC=0.9712) mis de côté !
Epoch 02/10 -> loss: 0.0916 | AUC: 0.9755
   [+] Nouveau meilleur modèle (AUC=0.9755) mis de côté !
Epoch 03/10 -> loss: 0.0787 | AUC: 0.9869
   [+] Nouveau meilleur modèle (AUC=0.9869) mis de côté !
Epoch 04/10 -> loss: 0.0620 | AUC: 0.9893
   [+] Nouveau meilleur modèle (AUC=0.9893) mis de côté !
Epoch 05/10 -> loss: 0.0488 | AUC: 0.9892
Epoch 06/10 -> loss: 0.0370 | AUC: 0.9913
   [+] Nouveau meilleur modèle (AUC=0.9913) mis de côté !
Epoch 07/10 -> loss: 0.0319 | AUC: 0.9934
   [+] Nouveau meilleur modèle (AUC=0.9934) mis de côté !
Epoch 08/10 -> loss: 0.0210 | AUC: 0.9945
   [+] Nouveau meilleur modèle (AUC=0.9945) mis de côté !
Epoch 09/10 -> loss: 0.0161 | AUC: 0.9946
   [+] Nouveau meilleur modèle (AUC=0.9946) mis de côté !
Epoch 10/10 -> loss: 0.0143 | AUC: 0.9946
   [+] No

In [22]:
print('Chargement du modèle pré-entrainé...')
model = GWClassifier().to(DEVICE)
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()

with h5py.File(TEST_H5, 'r') as f:
    num_test = f['X'].shape[0]

test_loader = DataLoader(
    GWDataset(TEST_H5, np.arange(num_test), mode='test'), 
    BATCH_SIZE, 
    shuffle=False
)

all_probs, all_ids = [], []
with torch.no_grad():
    for imgs, batch_ids in test_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
        all_probs.extend(probs)
        all_ids.extend(batch_ids)

preds = (np.array(all_probs) >= 0.5).astype(int)
submission = pd.DataFrame({'id': all_ids, 'target': preds})
submission.to_csv('submission.csv', index=False)

print('C\'est terminé ! submission.csv est prêt.')
submission.head(10)

Chargement du modèle pré-entrainé...
C'est terminé ! submission.csv est prêt.


,id,target
0,gw_test_00000,0
1,gw_test_00001,0
2,gw_test_00002,0
3,gw_test_00003,0
4,gw_test_00004,0
5,gw_test_00005,0
6,gw_test_00006,1
7,gw_test_00007,0
8,gw_test_00008,0
9,gw_test_00009,0


In [25]:
from IPython.display import FileLink
FileLink("submission.csv")

/kaggle/working/submission.csv